# 🔌 Insulator Mechanical Strength Analysis

**Author:** R VISHAK NAIR  
**Dataset:** `insulator_strength.csv`  
**Goal:** Analyze factors that affect the mechanical strength of electrical insulators, including aging, leakage current, core diameter, and material density.

---

## 📌 Table of Contents
1. [Setup & Data Loading](#1)
2. [Aging Duration vs Mechanical Strength](#2)
3. [Leakage Current Analysis](#3)
4. [Material Density & Strength Efficiency](#4)
5. [Core Diameter vs Mechanical Strength](#5)
6. [Combined Risk: High Leakage + Low Strength](#6)
7. [Remaining Strength Estimation](#7)
8. [Correlation Summary](#8)
9. [Key Findings & Recommendations](#9)

---
## 1. Setup & Data Loading <a id='1'></a>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# ── Load Data ──────────────────────────────────────────────────────────────
# Update the path below to match your local file location
df = pd.read_csv(r"C:\Users\Vishak\Documents\Project\insulator strength analysis\insulator_strength (1).csv")

# Round continuous columns to whole numbers for cleaner grouping
df['Aging_Duration'] = df['Aging_Duration'].round(0)
df['Core_Diameter']  = df['Core_Diameter'].round(0)

print(f"Dataset shape: {df.shape}")
df.head(3)

---
## 2. Aging Duration vs Mechanical Strength <a id='2'></a>

**Question:** Does mechanical strength decrease as insulators age?

We group by `Aging_Duration` and compute the average `Mechanical_Strength` at each time point.

In [ ]:
!pip install --upgrade nbformat

In [ ]:


# 1. Compute the average, but keep it as a DataFrame using .reset_index()
aging_analysis = df.groupby("Aging_Duration")["Mechanical_Strength"].mean().reset_index()

# 2. Create the interactive line plot with Plotly Express
fig = px.line(
    aging_analysis,
    x="Aging_Duration",
    y="Mechanical_Strength",
    markers=True,  # This adds the "o" markers to your line
    title="Average Mechanical Strength vs Aging Duration",
    labels={
        "Aging_Duration": "Aging Duration (years)",
        "Mechanical_Strength": "Avg Mechanical Strength (kN)"
    },
    template="plotly_white"  # Gives it a clean, modern look with grids built-in
)

# 3. Update the line color to steelblue
fig.update_traces(line_color="steelblue")

# 4. Display the plot right inside your PyCharm Jupyter Notebook
fig.show()


**Finding:** A downward trend in strength as aging duration increases confirms material degradation over time.  
Insulators with higher aging durations should be prioritized for inspection or replacement.

---
## 3. Leakage Current Analysis <a id='3'></a>

**Question:** Do insulators with higher leakage current show lower mechanical strength?

We classify insulators into three groups — High, Medium, Low leakage — using the 25th and 75th percentiles as thresholds.

In [ ]:
high_threshold = df["Leakage_Current"].quantile(0.75)
low_threshold  = df["Leakage_Current"].quantile(0.25)

def classify_leakage(value):
    if value > high_threshold:
        return "High Leakage"
    elif value < low_threshold:
        return "Low Leakage"
    else:
        return "Medium Leakage"

df["Leakage_Status"] = df["Leakage_Current"].apply(classify_leakage)

leakage_strength = df.groupby("Leakage_Status")["Mechanical_Strength"].mean().sort_values().reset_index()


fig2 = px.bar(
    leakage_strength,
    x = "Leakage_Status",
    y = "Mechanical_Strength",
    color = "Leakage_Status",
    labels={
        "Leakage_Status": "Leakage Status",
        "Mechanical_Strength": "Average Mechanical Strength"},
    template="plotly_white"
)

fig2.show()
print(leakage_strength)

**Finding:** Insulators classified as "High Leakage" tend to have lower average mechanical strength.  
This suggests leakage current is a reliable stress indicator and can be used as an early warning signal.

---
## 4. Material Density & Strength Efficiency <a id='4'></a>

**Question:** Does denser material produce stronger insulators relative to their core size?

We define **Strength Efficiency** as `Mechanical_Strength / Core_Diameter`, then compare the bottom 25% vs top 25% of material density.

In [ ]:
df['Strength_Efficiency'] = (df['Mechanical_Strength'] / df['Core_Diameter']).round(4)

low_density_threshold  = df['Material_Density'].quantile(0.25)
high_density_threshold = df['Material_Density'].quantile(0.75)

low_eff  = df[df['Material_Density'] <= low_density_threshold]['Strength_Efficiency'].mean()
high_eff = df[df['Material_Density'] >= high_density_threshold]['Strength_Efficiency'].mean()

print(f"Efficiency — Low Density  (≤25th pct): {low_eff:.4f} kN/mm")
print(f"Efficiency — High Density (≥75th pct): {high_eff:.4f} kN/mm")


fig3 = px.bar(
    x=["Low Density", "High Density"],
    y=[low_eff, high_eff],
    title="Strength Efficiency over Material Density",
    labels={
        "x": "Density Group",
        "y": "Strength Efficiency (kN/mm)"
    },
    color=["Low Density", "High Density"]
)

fig3.show()

# Visual comparison


**Finding:** Higher-density materials show greater strength efficiency — meaning they deliver more mechanical strength per mm of core diameter.  
This supports using denser materials in high-stress environments.

---
## 5. Core Diameter vs Mechanical Strength <a id='5'></a>

**Question:** Do larger core diameters lead to greater mechanical strength?

We also flag potentially **inefficient designs** — those with above-average core diameter but below-average mechanical strength.

In [ ]:
# ── Average strength by diameter ───────────────────────────────────────────
diameter_strength = df.groupby("Core_Diameter")["Mechanical_Strength"].mean()

import pandas as pd
import plotly.express as px

# 1. Group by Core_Diameter and calculate the average Mechanical Strength
diameter_strength = df.groupby("Core_Diameter")["Mechanical_Strength"].mean().reset_index()

# 2. Plot using Plotly Express
fig5 = px.line(
    diameter_strength,
    x="Core_Diameter",
    y="Mechanical_Strength",
    title="<b>Average Mechanical Strength by Core Diameter</b>",
    labels={
        "Core_Diameter": "Core Diameter (mm)",
        "Mechanical_Strength": "Avg Mechanical Strength (kN)"
    },
    markers=True
)

# 3. Customize layout for better readability
fig5.update_layout(
    title_x=0.5,
    template="plotly_white",
    xaxis=dict(dtick=1), # Shows every integer step if diameters are whole numbers
    yaxis_title="Avg Mechanical Strength ($\text{kN}$)",
    xaxis_title="Core Diameter ($\text{mm}$)"
)

# Display the interactive chart
fig.show(renderer="notebook")

# ── Correlation ─────────────────────────────────────────────────────────────
corr_diameter = df["Core_Diameter"].corr(df["Mechanical_Strength"])
print(f"Correlation between Core Diameter and Mechanical Strength: {corr_diameter:.4f}")

# ── Inefficient designs ─────────────────────────────────────────────────────
inefficient = df[
    (df["Core_Diameter"] > df["Core_Diameter"].mean()) &
    (df["Mechanical_Strength"] < df["Mechanical_Strength"].mean())
]
print(f"\nInefficient design count (large core, weak strength): {len(inefficient)}")
inefficient.head()

**Finding:** A larger core diameter does not always guarantee higher strength.  
Insulators with oversized cores but weak strength may point to material quality issues or manufacturing defects.

---
## 6. Combined Risk: High Leakage + Low Strength <a id='6'></a>

**Question:** What percentage of insulators are doubly at-risk — high leakage AND low mechanical strength?

These units represent the most critical failure candidates.

In [ ]:
low_strength_threshold = df['Mechanical_Strength'].quantile(0.25)

at_risk = df[
    (df['Leakage_Status'] == 'High Leakage') &
    (df['Mechanical_Strength'] < low_strength_threshold)
]

pct_at_risk = (len(at_risk) / len(df)) * 100

print(f"At-risk insulators (High Leakage + Low Strength): {len(at_risk)}")
print(f"Percentage of total dataset: {pct_at_risk:.2f}%")
at_risk.head()

**Finding:** A measurable percentage of insulators meet both risk criteria simultaneously.  
These should be the highest priority for preventive maintenance or replacement.

---
## 7. Remaining Strength Estimation <a id='7'></a>

**Business Question:** Which insulators have the least remaining strength?

We estimate a **Remaining Strength** score using the formula:  
`Remaining_Strength = 100 − (Leakage_Current × Aging_Duration)`  

The 5 insulators with the lowest scores are most urgent.

In [ ]:
df["Remaining_Strength"] = 100 - (df["Leakage_Current"] * df["Aging_Duration"])

critical_5 = df.nsmallest(5, "Remaining_Strength")[
    ["Aging_Duration", "Leakage_Current", "Mechanical_Strength", "Remaining_Strength"]
]

print("Top 5 Most Critical Insulators:")
print(critical_5.to_string(index=False))

**Finding:** The formula flags the most degraded insulators. These 5 records should be inspected immediately.  
> ⚠️ Note: This is a simplified heuristic. A proper degradation model would use domain-validated formulas.

---
## 8. Correlation Summary <a id='8'></a>

Let's compute the full correlation matrix for key numeric variables to summarize all relationships.

In [ ]:
cols = ["Aging_Duration", "Leakage_Current", "Mechanical_Strength", 
        "Core_Diameter", "Material_Density"]

corr_matrix = df[cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm",
            linewidths=0.5, square=True)
plt.title("Correlation Matrix — Key Variables")
plt.tight_layout()
plt.show()

print(corr_matrix)

**Finding:** The heatmap clearly shows which variables are positively or negatively associated with mechanical strength.  
Strong negative correlations with aging/leakage confirm our earlier findings.

---
## 9. Key Findings & Recommendations <a id='9'></a>

### ✅ What We Found

| # | Finding | Implication |
|---|---------|-------------|
| 1 | Mechanical strength **decreases with aging** | Implement age-based replacement schedules |
| 2 | **High leakage current = lower strength** | Use leakage as an early warning KPI |
| 3 | **High-density materials** are more efficient per mm of core | Prefer denser materials for critical deployments |
| 4 | **Large core ≠ strong insulator** — inefficient designs exist | Review manufacturing QA process |
| 5 | A subset of insulators have **both** high leakage and low strength | These need immediate attention |

ve Model** — Use `sklearn` to build a regression model predicting `Mechanical_Strength` from the other features.
2. **Clustering** — Apply K-Means clustering to segment insulators into risk tiers automatically.
3. **Time-Series** — If timestamps exist, model degradation curves per insulator over time.
4. **Dashboard** — Build an interactive dashboard using `Plotly` or `Streamlit` to let operators filter and explore live.
5. **Anomaly Detection** — Flag statistical outliers using IQR or Isolation Forest.